## delicious_grab_zone_log.csv file upload

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Delicious Grab Zone").getOrCreate()
df=spark.read.option("header","true").option("inferschema","true").csv("/Volumes/workspace/delicious_grab_zone_schema/delicious_grab_zone_volume/delicious_grab_zone_log.csv")


In [0]:
df.display()

Order id,date,customer name,item,quantity,price,address,payment mode
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card


## **Data Cleaning**

In [0]:
#Remove Duplicates
df_clean=df.dropDuplicates()
print(df_clean.count())

100


In [0]:
#Handling Null Values
#Whether Null or Not
from pyspark.sql.functions import *
df_clean=df.select([col(c).isNull().alias(c) for c in df.columns])
df_clean.show()

+--------+-----+-------------+-----+--------+-----+-------+------------+
|Order id| date|customer name| item|quantity|price|address|payment mode|
+--------+-----+-------------+-----+--------+-----+-------+------------+
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|  false|       false|
|   false|false|        false|false|   false|false|

In [0]:
#If exist
df_clean = df.na.drop()  
df_clean = df.na.fill({"payment mode": "Unknown"})


In [0]:
#Correcting Datatype 
#Recheck the data type
df_clean = df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
display(df_clean)

Order id,date,customer name,item,quantity,price,address,payment mode
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card


In [0]:
#Standardize column names 
df_clean =(df.withColumnRenamed("Order id", "Customer_id") 
             .withColumnRenamed("date","Delivery_date")
             .withColumnRenamed("customer name","Customer_name")
             .withColumnRenamed("item", "Item")
             .withColumnRenamed("quantity", "Quantity")
             .withColumnRenamed("price", "Price")
             .withColumnRenamed("address", "Address")
             .withColumnRenamed("payment mode", "Mode_of_payment"))
display(df_clean)


Customer_id,Delivery_date,Customer_name,Item,Quantity,Price,Address,Mode_of_payment
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card


In [0]:
#Creating new file for cleaned data
df_clean.write.mode("overwrite").csv("/Volumes/workspace/delicious_grab_zone_schema/delicious_grab_zone_volume/delicious_grab_zone_log_clean.csv", header=True)
